# Kaggle training — Akkadian → English (ByT5)

Вариант для Kaggle Notebooks (квота 30 GPU-ч/нед, сессии до 12 ч). Рассчитан на
**Save & Run All (Commit)** — весь ноутбук выполняется батчем, `results.csv` появится в Output.
Чекпоинты пушатся на HF Hub, резюм после обрыва — автоматический.

**Настройка (один раз):**
1. Справа **Session options → Accelerator → GPU P100** (или T4 x2) и **Internet → On**
   (для интернета нужен подтверждённый телефон в профиле Kaggle).
2. Справа **Input → + Add Input** → вкладка Competitions → **Deep Past Initiative: Machine Translation**.
3. Меню **Add-ons → Secrets** → добавь `HF_TOKEN` (тип Write) и `WANDB_API_KEY`, включи Attach.

**Два режима** (переключатель `TRAIN` в первой ячейке):
- `TRAIN=True` — обучить модель по `CONFIG`, затем оценить и собрать сабмит.
- `TRAIN=False` — только оценка + сабмит уже обученной модели с Hub (без переобучения, ~20 мин).

In [ ]:
CONFIG = "configs/baseline.yaml"  # <- какой эксперимент
TRAIN = True                       # <- False = только оценка+сабмит модели с Hub
BRANCH = "ml-dev"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # на T4 x2 учимся на одной GPU
!nvidia-smi -L

In [ ]:
!rm -rf /kaggle/working/repo
!git clone --branch {BRANCH} https://github.com/ObjoradDdd/ml-hits-3-lab.git /kaggle/working/repo
%cd /kaggle/working/repo/ml
!pip install -q -e . sacrebleu

In [ ]:
# секреты -> окружение; логинимся в HF (нужно для приватного репозитория с весами)
import yaml
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = secrets.get_secret(key)
    except Exception:
        print("no secret:", key)

from huggingface_hub import login, whoami
login(os.environ["HF_TOKEN"])
HF_USER = whoami()["name"]
RUN_NAME = yaml.safe_load(open(CONFIG))["run_name"]
HUB_ID = f"{HF_USER}/akkadian-{RUN_NAME}"
FINAL = yaml.safe_load(open(CONFIG))["output_dir"] + "/final"
# что грузим на оценке/сабмите: локальную обученную модель или модель с Hub
MODEL = FINAL if TRAIN else HUB_ID
print("HUB_ID:", HUB_ID, "| оценка/сабмит из:", MODEL)

In [ ]:
# данные соревнования уже подключены как Input — копируем нужные csv
COMP = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
!mkdir -p data && cp {COMP}/train.csv {COMP}/test.csv \
    {COMP}/published_texts.csv {COMP}/Sentences_Oare_FirstWord_LinNum.csv data/
!python -m akkadian_nmt.data_prep --data_dir=./data --out_dir=./data/processed

In [ ]:
# обучение (только если TRAIN=True); резюм с Hub после обрыва — автоматический
if TRAIN:
    !python -m akkadian_nmt.train --config={CONFIG} \
        --push_to_hub=True --hub_model_id={HUB_ID}
else:
    print("TRAIN=False — пропускаем обучение, берём модель с Hub")

In [ ]:
# оценка на испорченном dev: greedy vs beam {1,4,8} (эксперимент 2)
!python -m akkadian_nmt.evaluate beam_sweep --model_dirs={MODEL} --max_samples=200

In [ ]:
# предсказания для сабмита -> /kaggle/working/submission.csv
# (вкладка Output -> скачать или Submit to competition)
# делаем ДО COMET, пока в окружении правильный transformers==5.10.1
!python model.py predict-file --dataset=data/test.csv --model_dir={MODEL}
!cp data/results.csv /kaggle/working/submission.csv
!head -3 /kaggle/working/submission.csv

In [ ]:
# полный метрический набор (BLEU + chrF++ + geo-mean + COMET) — beam=4
# ВАЖНО: запускаем ПОСЛЕДНИМ — unbabel-comet тянет transformers<5.0 и портит окружение
!pip install -q unbabel-comet
!python -m akkadian_nmt.evaluate run --model_dirs={MODEL} --num_beams=4 --comet=True \
    --out_file=data/dev_predictions.json